# 🌐 Web Intelligence Pipeline — Chris Lake Edition
**Legion-Jacked-Pipeline | Sovereign Edge Vector Augmentation**

3-Lane Delta Architecture applied to live web API data.

| Lane | Store | Purpose |
|------|-------|---------|
| **Lane 1** | DuckDB (WebDB) | Raw API JSON — immutable physical truth |
| **Lane 2** | LanceDB (VectorDB) | Snowflake Arctic 1024-dim augmented vectors |
| **Lane 3** | H.O.R.N. Logs | Structured audit events |

> Data ingest and vector embedding are fully separated from the LLM reasoning cell.


## ⚙️ CELL 1 — Imports & Configuration

In [ ]:
import json
import os
import sys
import time
import warnings

warnings.filterwarnings("ignore")
sys.stdout.reconfigure(encoding="utf-8")

from datetime import datetime
from typing import List, Optional

import duckdb
import httpx
import lancedb
import ollama
import pyarrow as pa
import spotipy
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from spotipy.oauth2 import SpotifyClientCredentials

load_dotenv()

# ── Sovereign Edge Config ──────────────────────────────────────
OLLAMA_HOST        = "http://127.0.0.1:11434"
OLLAMA_EMBED_MODEL = "snowflake-arctic-embed:latest"

DB_PATH      = "./web_intel_sonicdb.duckdb"
LANCEDB_PATH = "./lancedb_web_intel_rag"
TABLE_NAME   = "chris_lake_web_intel"

ARTIST_NAME     = "Chris Lake"
CHRIS_LAKE_ID   = "5Igpc9iLZ3YGtKeYfSrrOE"  # Spotify ID
MB_HEADERS      = {"User-Agent": "LegionJackedPipeline/1.0 (research@legionjacked.ai)"}

sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=os.getenv("SPOTIFY_CLIENT_ID"),
    client_secret=os.getenv("SPOTIFY_CLIENT_SECRET")
))

print(f"Config loaded. Embed model: {OLLAMA_EMBED_MODEL}")
print(f"DuckDB  : {DB_PATH}")
print(f"LanceDB : {LANCEDB_PATH}")


## 🏗️ CELL 2 — Pydantic Schema Definitions (3-Lane Delta)

In [ ]:
class Lane1WebRawPayload(BaseModel):
    """Lane 1: Immutable raw web API payload — physical truth"""
    source:        str
    artist_name:   str
    release_title: str
    release_date:  Optional[str] = None
    album_type:    Optional[str] = None
    label:         Optional[str] = None
    total_tracks:  Optional[int] = None
    genre:         Optional[str] = None
    style:         Optional[str] = None
    isrc:          Optional[str] = None
    track_list:    Optional[str] = None   # JSON string of track names
    raw_json:      str
    ingested_at:   str = Field(default_factory=lambda: datetime.now().isoformat())

class Lane2AugmentedVector(BaseModel):
    """Lane 2: Combined multi-source payload ready for Snowflake embedding"""
    release_title: str
    artist_name:   str
    release_date:  Optional[str] = None
    label:         Optional[str] = None
    genre:         Optional[str] = None
    style:         Optional[str] = None
    isrc:          Optional[str] = None
    album_type:    Optional[str] = None
    embed_text:    str
    vector:        Optional[List[float]] = None

print("3-Lane Delta Pydantic schemas loaded.")


## 🗄️ CELL 3 — Lane 1: DuckDB WebDB Initialization

In [ ]:
conn = duckdb.connect(DB_PATH)

conn.execute("""
    CREATE TABLE IF NOT EXISTS web_intel_raw (
        id            INTEGER,
        source        TEXT,
        artist_name   TEXT,
        release_title TEXT,
        release_date  TEXT,
        album_type    TEXT,
        label         TEXT,
        total_tracks  INTEGER,
        genre         TEXT,
        style         TEXT,
        isrc          TEXT,
        track_list    TEXT,
        raw_json      TEXT,
        ingested_at   TEXT
    )
""")

existing = conn.execute("SELECT COUNT(*) FROM web_intel_raw").fetchone()[0]
print(f"WebDB initialized: {DB_PATH}")
print(f"Existing rows    : {existing}")


## 🎵 CELL 4 — Data Ingest: Spotify Discography

In [ ]:
spotify_payloads = []

print(f"Fetching Spotify discography for {ARTIST_NAME}...")
albums = sp.artist_albums(CHRIS_LAKE_ID, limit=10)

for a in albums["items"]:
    try:
        full = sp.album(a["id"])
        tracks = [t["name"] for t in full["tracks"]["items"]]
        payload = Lane1WebRawPayload(
            source        = "spotify",
            artist_name   = ARTIST_NAME,
            release_title = a["name"],
            release_date  = a.get("release_date"),
            album_type    = a.get("album_type"),
            label         = full.get("label", ""),
            total_tracks  = full.get("total_tracks"),
            track_list    = json.dumps(tracks),
            raw_json      = json.dumps(a)
        )
        spotify_payloads.append(payload)
        print(f"  [{payload.release_date}] {payload.release_title:<40} | {payload.label}")
    except Exception as e:
        print(f"  Skipped {a.get('name','?')}: {e}")

print(f"\nSpotify payloads ready: {len(spotify_payloads)}")


## 🎛️ CELL 5 — Data Ingest: Discogs Genre + Style

In [ ]:
discogs_payloads = []
all_genres = set()
all_styles = set()

print(f"Fetching Discogs releases for {ARTIST_NAME}...")
r = httpx.get(
    "https://api.discogs.com/database/search",
    params={"artist": ARTIST_NAME, "type": "release", "per_page": 10},
    headers={"User-Agent": "LegionJackedPipeline/1.0"},
    timeout=15
)

for rel in r.json().get("results", []):
    genres = rel.get("genre", [])
    styles = rel.get("style", [])
    all_genres.update(genres)
    all_styles.update(styles)
    
    payload = Lane1WebRawPayload(
        source        = "discogs",
        artist_name   = ARTIST_NAME,
        release_title = rel.get("title", ""),
        release_date  = str(rel.get("year", "")),
        genre         = ", ".join(genres),
        style         = ", ".join(styles),
        raw_json      = json.dumps(rel)
    )
    discogs_payloads.append(payload)
    print(f"  {str(rel.get('year','?')):<6} {rel.get('title','')[:40]:<40} | {', '.join(styles[:2])}")

print(f"\nDiscogs payloads ready : {len(discogs_payloads)}")
print(f"Genre cluster detected : {sorted(all_genres)}")
print(f"Style cluster detected : {sorted(all_styles)}")


## 🎼 CELL 6 — Data Ingest: MusicBrainz ISRCs

In [ ]:
mb_payloads = []

print(f"Fetching MusicBrainz recordings for {ARTIST_NAME}...")

r = httpx.get(
    "https://musicbrainz.org/ws/2/artist/",
    params={"query": ARTIST_NAME, "fmt": "json", "limit": 1},
    headers=MB_HEADERS, timeout=15
)
mb_artist = r.json()["artists"][0]
mb_id = mb_artist["id"]
print(f"MBID: {mb_id} | Country: {mb_artist.get('country', 'N/A')}")

time.sleep(1)  # MusicBrainz rate limit: 1 req/sec

r2 = httpx.get(
    "https://musicbrainz.org/ws/2/recording/",
    params={"artist": mb_id, "fmt": "json", "limit": 20, "inc": "isrcs"},
    headers=MB_HEADERS, timeout=15
)
recordings = r2.json().get("recordings", [])

for rec in recordings:
    isrcs = rec.get("isrcs", [])
    payload = Lane1WebRawPayload(
        source        = "musicbrainz",
        artist_name   = ARTIST_NAME,
        release_title = rec["title"],
        isrc          = isrcs[0] if isrcs else None,
        raw_json      = json.dumps(rec)
    )
    mb_payloads.append(payload)
    if isrcs:
        print(f"  {rec['title'][:45]:<45} | ISRC: {isrcs[0]}")

print(f"\nMusicBrainz payloads ready: {len(mb_payloads)}")


## 💾 CELL 7 — Lane 1 Flush: All Sources → DuckDB WebDB

In [ ]:
all_payloads = spotify_payloads + discogs_payloads + mb_payloads

# Clear existing data for this artist to avoid duplicates on re-runs
conn.execute("DELETE FROM web_intel_raw WHERE artist_name = ?", [ARTIST_NAME])

for i, p in enumerate(all_payloads):
    conn.execute("""
        INSERT INTO web_intel_raw
        (id, source, artist_name, release_title, release_date, album_type,
         label, total_tracks, genre, style, isrc, track_list, raw_json, ingested_at)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, [i, p.source, p.artist_name, p.release_title, p.release_date,
             p.album_type, p.label, p.total_tracks, p.genre, p.style,
             p.isrc, p.track_list, p.raw_json, p.ingested_at])

conn.commit()
print(f"Flushed {len(all_payloads)} raw payloads to DuckDB WebDB (Lane 1)")
print()

# Quick audit
df = conn.execute("""
    SELECT source, COUNT(*) as rows, COUNT(isrc) as with_isrc
    FROM web_intel_raw
    WHERE artist_name = ?
    GROUP BY source
""", [ARTIST_NAME]).df()
print(df.to_string(index=False))


## ❄️ CELL 8 — Lane 2: Snowflake Arctic Embedding
> Data ingest only — Ollama reasoning test is in its own separate cell below.

In [ ]:
ollama_client = ollama.Client(host=OLLAMA_HOST)

def embed(text: str) -> List[float]:
    r = ollama_client.embeddings(model=OLLAMA_EMBED_MODEL, prompt=text)
    return r["embedding"]

# Pull all spotify releases from DuckDB — these are the primary vector targets
rows = conn.execute("""
    SELECT release_title, release_date, album_type, label, total_tracks, track_list
    FROM web_intel_raw
    WHERE source = 'spotify' AND artist_name = ?
""", [ARTIST_NAME]).fetchall()

# Build genre/style string from Discogs (cross-source augmentation)
genre_row = conn.execute("""
    SELECT STRING_AGG(DISTINCT genre, ', '), STRING_AGG(DISTINCT style, ', ')
    FROM web_intel_raw
    WHERE source = 'discogs' AND artist_name = ?
""", [ARTIST_NAME]).fetchone()
genre_str = genre_row[0] or ""
style_str = genre_row[1] or ""

print(f"Routing {len(rows)} releases to Snowflake Arctic on {OLLAMA_HOST}...")

augmented_vectors = []
for row in rows:
    title, release_date, album_type, label, total_tracks, track_list_json = row
    tracks = json.loads(track_list_json) if track_list_json else []

    # Find matching ISRC from MusicBrainz
    isrc_row = conn.execute("""
        SELECT isrc FROM web_intel_raw
        WHERE source='musicbrainz' AND artist_name=? AND isrc IS NOT NULL
        AND LOWER(release_title) LIKE ?
        LIMIT 1
    """, [ARTIST_NAME, f"%{title[:10].lower()}%"]).fetchone()
    isrc = isrc_row[0] if isrc_row else ""

    embed_text = (
        f"artist: {ARTIST_NAME} "
        f"release: {title} "
        f"release_date: {release_date} "
        f"album_type: {album_type} "
        f"label: {label} "
        f"total_tracks: {total_tracks} "
        f"genre: {genre_str} "
        f"style: {style_str} "
        + (f"isrc: {isrc} " if isrc else "")
        + (f"tracks: {' | '.join(tracks[:5])} " if tracks else "")
    )

    vector = embed(embed_text)

    augmented_vectors.append({
        "release_title": title,
        "artist_name":   ARTIST_NAME,
        "release_date":  release_date or "",
        "album_type":    album_type or "",
        "label":         label or "",
        "genre":         genre_str,
        "style":         style_str,
        "isrc":          isrc,
        "embed_text":    embed_text,
        "vector":        vector
    })
    print(f"  Embedded: {title[:45]:<45} | dim={len(vector)}")

print(f"\nAll {len(augmented_vectors)} vectors generated.")


## 🗃️ CELL 9 — Lane 2 Flush: Augmented Vectors → LanceDB

In [ ]:
db = lancedb.connect(LANCEDB_PATH)

schema = pa.schema([
    pa.field("release_title", pa.string()),
    pa.field("artist_name",   pa.string()),
    pa.field("release_date",  pa.string()),
    pa.field("album_type",    pa.string()),
    pa.field("label",         pa.string()),
    pa.field("genre",         pa.string()),
    pa.field("style",         pa.string()),
    pa.field("isrc",          pa.string()),
    pa.field("embed_text",    pa.string()),
    pa.field("vector",        pa.list_(pa.float32(), 1024)),
])

if TABLE_NAME in db.table_names():
    db.drop_table(TABLE_NAME)

table = db.create_table(TABLE_NAME, data=augmented_vectors, schema=schema)

print(f"Vectors flushed to LanceDB: {LANCEDB_PATH}")
print(f"Table '{TABLE_NAME}': {table.count_rows()} rows")


## 📊 CELL 10 — Lane 1 Audit: DuckDB Analytics Report

In [ ]:
print("WebDB Lane 1 Audit Report")
print("=" * 60)

df = conn.execute("""
    SELECT
        source,
        COUNT(*) as payloads,
        MIN(release_date) as earliest,
        MAX(release_date) as latest,
        COUNT(isrc) as with_isrc,
        COUNT(genre) as with_genre
    FROM web_intel_raw
    WHERE artist_name = ?
    GROUP BY source
    ORDER BY source
""", [ARTIST_NAME]).df()

print(df.to_string(index=False))
print()
print(f"Total archived payloads: {conn.execute('SELECT COUNT(*) FROM web_intel_raw WHERE artist_name=?', [ARTIST_NAME]).fetchone()[0]}")
conn.close()


## 🔍 CELL 11 — Lane 2 Query: Semantic Similarity Search

In [ ]:
def query(q: str, top_k: int = 3):
    v = embed(q)
    tbl = db.open_table(TABLE_NAME)
    results = tbl.search(v).limit(top_k).to_list()
    print(f"Query: '{q}'")
    print("-" * 50)
    for i, r in enumerate(results):
        print(f"  [{i+1}] {r['release_title']}")
        print(f"       Date: {r['release_date']} | Type: {r['album_type']} | Genre: {r['genre'][:40]}")
    print()

query("tech house deep electronic single")
query("remix collaborative collab")


## 🧠 CELL 12 — Ollama Reasoning Test (Standalone)
> This cell is intentionally isolated from the data ingest pipeline above. Run independently.

In [ ]:
# ── STANDALONE OLLAMA REASONING TEST ──────────────────────────
# Pull the top 3 most recent releases from LanceDB and ask Gemma to reason over them.

tbl = db.open_table(TABLE_NAME)
top_releases = tbl.search(embed("electronic music recent release")).limit(3).to_list()

context = "\n".join([
    f"- {r['release_title']} ({r['release_date']}) | Genre: {r['genre']} | Style: {r['style']}"
    for r in top_releases
])

prompt = f"""
You are a music intelligence analyst. Based on these Chris Lake releases:

{context}

What patterns do you observe in his recent output? What direction is his sound heading?
Keep it concise, max 3 sentences.
"""

print("Sending to Ollama LLM reasoning layer...")
print("=" * 60)

gemma_client = ollama.Client(host=OLLAMA_HOST)
response = gemma_client.chat(
    model="gemma:2b",
    messages=[{"role": "user", "content": prompt}],
    options={"temperature": 0.0}
)
print(response["message"]["content"])
print("=" * 60)
